In [ ]:
import json
import numpy as np
import torch
from datasets import Dataset
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
from transformers import BitsAndBytesConfig

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
try:
    from huggingface_hub import login
    login(YOUR_HF_TOKEN)
except Exception as e:
    print("Hugging Face login skipped:", e)

In [3]:
def combine_text(row):
    # Collect all existing description parts in order
    desc_parts = [row.get(f"case_description_{i}", "") for i in range(1, 9) if row.get(f"case_description_{i}")]
    # Collect all existing justification parts in order
    just_parts = [row.get(f"justification_{i}", "") for i in range(1, 5) if row.get(f"justification_{i}")]
    
    # Prioritize description by putting it first
    return " ".join(desc_parts + just_parts)

In [ ]:
def prepare_dataset(path, type_of_jurisdiction):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if type_of_jurisdiction == "supreme":
        label_map = {"respins": 0, "admis": 1}
        data = [item for item in data if item['label'] != 'inadmisibil']
    else:
        label_map = {"vinovat": 0, "nevinovat": 1}

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        
        # --- PROMPT ENGINEERING START ---
        
        if type_of_jurisdiction == "supreme":
            just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
            instructional_prompt = (
                f"Decide verdictul acestui caz juridic:\n\n"
                f"DESCRIERE: {desc}\n"
                f"JUSTIFICARE: {just}\n\n"
                f"Ai de ales între două răspunsuri: admis sau respins.\n"
            )
        else:
            instructional_prompt = (
                f"Decide verdictul acestui caz juridic:\n\n"
                f"DESCRIERE: {desc}\n"
                f"Ai de ales între două răspunsuri: vinovat sau nevinovat.\n"
            )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })
    
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

In [5]:
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    results = clf_metrics.compute(predictions=predictions, references=labels)
    
    decoded_preds = [str(p) for p in predictions]
    decoded_labels = [str(l) for l in labels]
    
    results.update(rouge.compute(predictions=decoded_preds, references=decoded_labels))    
    return results

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SUPREME COURT DECISIONS

In [8]:
path = "data/supreme-court-data.json"
train_raw, val_raw = prepare_dataset(path, "supreme")

## BERT Base LLM

In [ ]:
model_name = "dumitrescustefan/bert-base-romanian-cased-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 2718.68 examples/s]


In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [ ]:
training_args = TrainingArguments(
    output_dir="results/supreme-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
50,0.082600,0.358774,0.877049,0.867257,0.907407,0.830508,0.877049,0.000000,0.877049,0.877049
100,0.043700,0.374188,0.901639,0.892857,0.943396,0.847458,0.901639,0.000000,0.901639,0.901639
150,0.034000,0.508571,0.868852,0.857143,0.905660,0.813559,0.868852,0.000000,0.868852,0.868852
200,0.152900,0.531301,0.885246,0.867925,0.978723,0.779661,0.885246,0.000000,0.885246,0.885246


TrainOutput(global_step=220, training_loss=0.09186268286271529, metrics={'train_runtime': 55.6556, 'train_samples_per_second': 61.988, 'train_steps_per_second': 3.953, 'total_flos': 907733140992000.0, 'train_loss': 0.09186268286271529, 'epoch': 5.0})

## LLAMA

In [23]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 3023.26 examples/s]


In [42]:
device_map = {"": 0}

# Model Loading with LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.71s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,1.880200,1.378425,0.532787,0.424242,0.525000,0.355932,0.532787,0.000000,0.532787,0.532787
300,1.618400,1.217533,0.549180,0.529915,0.534483,0.525424,0.549180,0.000000,0.549180,0.549180
450,1.329500,1.534886,0.532787,0.344828,0.535714,0.254237,0.532787,0.000000,0.532787,0.532787
600,1.453300,1.395370,0.524590,0.325581,0.518519,0.237288,0.524590,0.000000,0.524590,0.524590
750,1.339400,1.391482,0.532787,0.359551,0.533333,0.271186,0.532787,0.000000,0.532787,0.532787
900,1.317200,1.143965,0.565574,0.475248,0.571429,0.406780,0.565574,0.000000,0.565574,0.565574
1050,1.050600,1.167608,0.540984,0.471698,0.531915,0.423729,0.549180,0.000000,0.540984,0.540984
1200,1.339900,1.197363,0.540984,0.461538,0.533333,0.406780,0.549180,0.000000,0.540984,0.540984
1350,1.128500,1.151453,0.565574,0.504673,0.562500,0.457627,0.565574,0.000000,0.565574,0.565574
1500,1.012700,1.195397,0.549180,0.455446,0.547619,0.389831,0.549180,0.000000,0.549180,0.549180


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=2070, training_loss=1.2777564597014643, metrics={'train_runtime': 1367.0645, 'train_samples_per_second': 1.514, 'train_steps_per_second': 1.514, 'total_flos': 4.120964619042816e+16, 'train_loss': 1.2777564597014643, 'epoch': 3.0})

# REGIONAL COURT DECISIONS

In [7]:
path = "data/regional-court-data.json"
train_raw, val_raw = prepare_dataset(path, "regional")

## BERT Base LLM

In [45]:
model_name = "dumitrescustefan/bert-base-romanian-cased-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:00<00:00, 1370.27 examples/s]


In [46]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [47]:
training_args = TrainingArguments(
    output_dir="results/regional-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
50,0.689000,0.662974,0.592036,0.010929,0.600000,0.005515,0.592036,0.000000,0.591285,0.592036
100,0.662100,0.617658,0.655898,0.418782,0.676230,0.303309,0.655522,0.000000,0.655898,0.655147
150,0.630900,0.658940,0.628099,0.241960,0.724771,0.145221,0.628099,0.000000,0.628099,0.627348
200,0.593700,0.618810,0.676935,0.417344,0.793814,0.283088,0.676935,0.000000,0.676183,0.676935
250,0.529800,0.511861,0.754320,0.691800,0.709865,0.674632,0.754320,0.000000,0.754320,0.754320
300,0.525800,0.510729,0.749812,0.729927,0.653120,0.827206,0.749812,0.000000,0.749812,0.749812
350,0.502800,0.513518,0.747558,0.706294,0.673333,0.742647,0.747558,0.000000,0.747558,0.747558
400,0.522300,0.491560,0.762585,0.689587,0.740506,0.645221,0.762209,0.000000,0.762585,0.762585
450,0.517300,0.521908,0.749812,0.609613,0.841424,0.477941,0.749812,0.000000,0.749812,0.750563
500,0.492600,0.458959,0.769346,0.690212,0.765101,0.628676,0.768595,0.000000,0.769346,0.769346


TrainOutput(global_step=2360, training_loss=0.3665808396319212, metrics={'train_runtime': 783.727, 'train_samples_per_second': 48.097, 'train_steps_per_second': 3.011, 'total_flos': 9917971231795200.0, 'train_loss': 0.3665808396319212, 'epoch': 5.0})

## LLAMA

In [8]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 1331/1331 [00:00<00:00, 1502.84 examples/s]


In [13]:
device_map = {"": 0}

# Model Loading with LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32,
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.56s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
training_args = TrainingArguments(
    output_dir="results/regional-llama-llm",
    learning_rate=2e-5,
    lr_scheduler_type="reduce_lr_on_plateau",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.1,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,1.457800,1.281135,0.496619,0.497751,0.420253,0.610294,0.496619,0.000000,0.495868,0.497370
300,1.372500,1.160414,0.523666,0.447735,0.425497,0.472426,0.523666,0.000000,0.522915,0.523666
450,1.190500,1.143307,0.543201,0.384615,0.427928,0.349265,0.542449,0.000000,0.543201,0.543201
600,1.294500,1.319359,0.569497,0.256809,0.436123,0.181985,0.568745,0.000000,0.569497,0.569497
750,1.430000,1.603319,0.574005,0.104265,0.370787,0.060662,0.574005,0.000000,0.573253,0.574005
900,1.656200,2.078074,0.585274,0.028169,0.333333,0.014706,0.584523,0.000000,0.584523,0.585274
1050,2.154800,1.917285,0.580766,0.041237,0.315789,0.022059,0.580766,0.000000,0.580766,0.581518
1200,2.024000,1.831824,0.590533,0.130781,0.493976,0.075368,0.589782,0.000000,0.590533,0.590533
1350,1.864400,1.695958,0.586777,0.159021,0.472727,0.095588,0.586777,0.000000,0.586777,0.586777
1500,1.972900,1.818734,0.592036,0.147567,0.505376,0.086397,0.592036,0.000000,0.592036,0.592036


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

KeyboardInterrupt: 